# Hands-On Workshop: Navigating the Big Data Analytics Lifecycle with R

**Duration:** ~2 hours | **Level:** Intermediate | **Language:** R (tidyverse / tidymodels)

## Welcome!

In this workshop we will walk through the **Big Data Analytics Lifecycle** end-to-end using a
realistic business dataset: the **Telco Customer Churn** dataset. You will move through five
phases that mirror how data science projects are run in industry:

| Phase | Focus | Time |
|---|---|---|
| 1 | Data Preparation & Hygiene | 25 min |
| 2 | Model Planning & EDA | 20 min |
| 3 | Model Building & Validation | 25 min |
| 4 | Communicating Results & Findings | 20 min |
| 5 | Operationalizing Analytics Models | 15 min |

Each phase includes:
- 📘 **Concept explanation** (theory + business relevance)
- 💻 **Worked example** (fully runnable code — just execute each cell in order)

### Prerequisites
- Basic familiarity with R and the `%>%` / `|>` pipe
- Basic understanding of classification models (logistic regression, decision trees)
- No prior `tidymodels` experience required — we'll explain as we go

### Dataset
We'll use the IBM **Telco Customer Churn** dataset, loaded directly from GitHub:
`https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv`

It contains ~7,043 customer records with demographic, account, and service-usage attributes, and
a `Churn` label indicating whether the customer left the company.


## Setup: Install & Load Packages

Run the cell below **once** to install any missing packages (skip if already installed in your
environment — e.g. most JupyterHub / RStudio Cloud R kernels used in class already have these).


In [ ]:
# ---- One-time setup: install required packages ----
# Uncomment the lines below if running for the first time on a fresh machine.

install.packages(c(
  "tidyverse",   # data wrangling & visualization (dplyr, ggplot2, readr, etc.)
  "tidymodels",  # unified modeling framework (recipes, parsnip, rsample, yardstick)
  "corrr",       # tidy correlation matrices + network plots
  "pROC",        # ROC curve / AUC utilities
  #"vip",         # variable importance plots
  "ranger",      # fast random forest engine used by tidymodels
  "plumber"      # turns R scripts into REST APIs (used in Phase 5)
))


In [ ]:
# ---- Load libraries ----
suppressPackageStartupMessages({
  library(tidyverse)   # dplyr, ggplot2, readr, purrr, tibble ...
  library(tidymodels)  # recipes, parsnip, rsample, workflows, yardstick
  library(corrr)       # tidy correlations
  library(pROC)        # ROC / AUC
  library(vip)         # variable importance plots
  library(ranger)       # random forest engine
})

# Reproducibility
set.seed(2026)

cat("Environment ready. R version:", R.version.string, "\n")


---
## Phase 1: Data Preparation & Hygiene (25 min)

### 📘 Concept

The **Big Data Analytics Lifecycle** always begins with **Data Preparation** — often called
ETL (Extract, Transform, Load) or "data hygiene." Industry surveys consistently show data
scientists spend **60–80% of project time** here. Why does it matter so much?

- **Garbage in, garbage out:** No model, however sophisticated, can compensate for dirty data.
- **Type errors compound:** A column read as text instead of numeric silently breaks every
  downstream calculation.
- **Missing values bias results:** Naively dropping or filling missing data can introduce bias
  that misleads business decisions (e.g., systematically under-representing a customer segment).
- **Business relevance:** For a telecom company, an unclean `Churn` dataset could lead to a
  retention model that misidentifies at-risk customers — a direct hit to revenue.

**Learning goals:** By the end of this phase you will be able to:
1. Load raw data and immediately profile it for quality issues.
2. Diagnose and fix a string-formatted numeric column.
3. Impute missing values responsibly (median imputation) and justify the choice.
4. Encode a categorical target variable for modeling.


In [ ]:
# ---- Load the raw dataset ----
data_url <- "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

churn_raw <- read_csv(data_url, show_col_types = FALSE)

glimpse(churn_raw)


In [ ]:
# ---- Diagnose data quality issues ----

# 1. How many missing values per column?
churn_raw %>%
  summarise(across(everything(), ~ sum(is.na(.)))) %>%
  pivot_longer(everything(), names_to = "column", values_to = "n_missing") %>%
  filter(n_missing > 0)

# 2. TotalCharges often arrives as character due to blank strings for new customers
#    with tenure = 0. Let's confirm:
class(churn_raw$TotalCharges)
sum(is.na(as.numeric(churn_raw$TotalCharges)))


In [ ]:
# ---- Instructor solution: clean & prepare the dataset ----

churn_clean <- churn_raw %>%
  # Fix TotalCharges: coerce to numeric (blank strings become NA automatically)
  mutate(TotalCharges = as.numeric(TotalCharges)) %>%
  # Median-impute the (small number of) resulting NAs in TotalCharges
  mutate(TotalCharges = if_else(
    is.na(TotalCharges),
    median(TotalCharges, na.rm = TRUE),
    TotalCharges
  )) %>%
  # Drop the customerID column - it's a unique identifier, not a predictive feature
  select(-customerID) %>%
  # Convert character columns to factors for modeling
  mutate(across(where(is.character), as.factor)) %>%
  # Encode the target as a factor with an explicit, model-friendly level order
  mutate(Churn = factor(Churn, levels = c("No", "Yes")))

glimpse(churn_clean)

# Sanity check: no more missing values
sum(is.na(churn_clean))


### 💻 Worked Example: Handle Missing Values & Filter Invalid Rows

An alternative cleaning strategy to the one above: instead of median-imputing `TotalCharges`,
this pipeline removes any row where `tenure == 0` **and** `TotalCharges` is still `NA` — these
represent brand-new customers with no billing history yet, a legitimate business reason to
exclude them from a churn-risk model rather than impute a fabricated value.


In [ ]:
# ---- Worked Example: Handle Missing Values & Filter Invalid Rows ----
churn_student <- churn_raw %>%
  mutate(TotalCharges = as.numeric(TotalCharges)) %>%
  # Filter out rows where tenure == 0 AND TotalCharges is NA
  filter(!(tenure == 0 & is.na(TotalCharges))) %>%
  # Convert all remaining character columns to factors
  mutate(across(where(is.character), as.factor))

# Check the result:
glimpse(churn_student)
sum(is.na(churn_student$TotalCharges))

---
## Phase 2: Model Planning & Exploratory Data Analysis (20 min)

### 📘 Concept

**Model Planning** is where we form hypotheses about what drives the outcome we care about and
decide which modeling approach is likely to work. This is guided by **Exploratory Data Analysis
(EDA)**: summary statistics, distributions, and correlations that reveal structure in the data
before we commit computational resources to model training.

- **Business relevance:** EDA often surfaces the "quick win" insight (e.g., "month-to-month
  contracts churn 3x more than two-year contracts") that stakeholders can act on immediately,
  even before a predictive model exists.
- **Statistical relevance:** Correlation analysis helps us spot multicollinearity and
  redundant features, and flags which numeric variables are worth engineering further.

**Learning goals:**
1. Profile a dataset using `glimpse()` and `summary()`.
2. Build and visualize a correlation matrix with `corrr`.
3. Use `ggplot2` to visualize a categorical driver of churn.


In [ ]:
# ---- Summary statistics ----
summary(churn_clean)


In [ ]:
# ---- Correlation matrix for numeric features ----
numeric_features <- churn_clean %>%
  select(where(is.numeric))

corr_matrix <- numeric_features %>%
  correlate(quiet = TRUE)

corr_matrix

# Visualize as a correlation network - stronger relationships get thicker/closer edges
corr_matrix %>%
  network_plot(min_cor = 0.1)


In [ ]:
# ---- Visualize: Churn rate by Contract Type ----
churn_clean %>%
  count(Contract, Churn) %>%
  group_by(Contract) %>%
  mutate(pct = n / sum(n)) %>%
  ggplot(aes(x = Contract, y = pct, fill = Churn)) +
  geom_col(position = "fill") +
  scale_y_continuous(labels = scales::percent) +
  labs(
    title = "Churn Rate by Contract Type",
    subtitle = "Month-to-month customers churn far more than longer-term contracts",
    x = "Contract Type", y = "Percentage of Customers", fill = "Churned?"
  ) +
  theme_minimal()


### 💻 Worked Example: Tenure Distribution by Churn

A `ggplot2` visualization showing the distribution of `tenure` (months as a customer),
segmented by `Churn`, using `geom_density()`.


In [ ]:
# ---- Worked Example: Tenure Distribution by Churn ----

ggplot(churn_clean, aes(x = tenure, fill = Churn)) +
  geom_density(alpha = 0.5) +
  labs(
    title = "Customer Tenure Distribution by Churn Status",
    subtitle = "Churned customers are concentrated at low tenure",
    x = "Tenure (months)", y = "Density"
  ) +
  theme_minimal()

---
## Phase 3: Model Building & Validation (25 min)

### 📘 Concept

With hypotheses in hand, we now build predictive models using the **`tidymodels`** framework,
which standardizes the workflow: **recipe** (preprocessing) → **model spec** → **workflow** →
**fit** → **evaluate**.

We'll compare two models:
- **Logistic Regression** — a simple, highly interpretable baseline. Great for explaining
  *why* a customer is at risk, in plain coefficients.
- **Random Forest** — an ensemble method that typically captures non-linear interactions
  (e.g., "long tenure + fiber optic + no tech support" as a combined risk factor) that logistic
  regression can miss.

We evaluate with **ROC-AUC** (overall discriminative ability), and **precision/recall** — the
metrics that matter most in churn, since the business cost of missing a churner (false negative)
is usually higher than the cost of a false alarm (false positive).

**Note on class imbalance:** Only ~27% of customers in this dataset churn. We'll keep this in
mind when interpreting accuracy (which can be misleadingly high on imbalanced data) — ROC-AUC
and recall are more trustworthy here.

**Learning goals:**
1. Split data with `rsample::initial_split()`.
2. Build a `recipe` and two model specifications.
3. Fit both models via `workflows` and generate class probabilities.
4. Compute and plot ROC curves / AUC.


In [ ]:
# ---- Train/test split (stratified on the outcome to preserve churn rate) ----
set.seed(2026)
churn_split <- initial_split(churn_clean, prop = 0.8, strata = Churn)
churn_train <- training(churn_split)
churn_test  <- testing(churn_split)

cat("Training rows:", nrow(churn_train), "| Test rows:", nrow(churn_test), "\n")
cat("Training churn rate:", round(mean(churn_train$Churn == "Yes"), 3), "\n")
cat("Test churn rate:    ", round(mean(churn_test$Churn == "Yes"), 3), "\n")


In [ ]:
# ---- Preprocessing recipe ----
# One recipe shared by both models: dummy-encode categoricals, normalize numerics.
churn_recipe <- recipe(Churn ~ ., data = churn_train) %>%
  step_dummy(all_nominal_predictors()) %>%
  step_zv(all_predictors()) %>%          # drop any zero-variance columns
  step_normalize(all_numeric_predictors())

# ---- Model specifications ----
log_reg_spec <- logistic_reg() %>%
  set_engine("glm") %>%
  set_mode("classification")

rf_spec <- rand_forest(trees = 500) %>%
  set_engine("ranger", importance = "impurity") %>%
  set_mode("classification")

# ---- Workflows ----
log_reg_wf <- workflow() %>%
  add_recipe(churn_recipe) %>%
  add_model(log_reg_spec)

rf_wf <- workflow() %>%
  add_recipe(churn_recipe) %>%
  add_model(rf_spec)


In [ ]:
# ---- Fit both models on the training set ----
log_reg_fit <- fit(log_reg_wf, data = churn_train)
rf_fit      <- fit(rf_wf, data = churn_train)

# ---- Generate class probabilities on the test set ----
log_reg_probs <- predict(log_reg_fit, churn_test, type = "prob") %>%
  bind_cols(predict(log_reg_fit, churn_test)) %>%
  bind_cols(churn_test %>% select(Churn))

rf_probs <- predict(rf_fit, churn_test, type = "prob") %>%
  bind_cols(predict(rf_fit, churn_test)) %>%
  bind_cols(churn_test %>% select(Churn))

head(log_reg_probs)


In [ ]:
# ---- ROC curves & AUC (using yardstick, tidymodels' metrics package) ----
log_reg_roc <- log_reg_probs %>%
  roc_curve(truth = Churn, .pred_Yes, event_level = "second") %>%
  mutate(model = "Logistic Regression")

rf_roc <- rf_probs %>%
  roc_curve(truth = Churn, .pred_Yes, event_level = "second") %>%
  mutate(model = "Random Forest")

bind_rows(log_reg_roc, rf_roc) %>%
  ggplot(aes(x = 1 - specificity, y = sensitivity, color = model)) +
  geom_path(linewidth = 1) +
  geom_abline(linetype = "dashed", color = "gray50") +
  labs(title = "ROC Curves: Logistic Regression vs. Random Forest",
       x = "False Positive Rate", y = "True Positive Rate", color = "Model") +
  theme_minimal()

# AUC values
log_reg_auc <- roc_auc(log_reg_probs, truth = Churn, .pred_Yes, event_level = "second")
rf_auc      <- roc_auc(rf_probs, truth = Churn, .pred_Yes, event_level = "second")

bind_rows(
  log_reg_auc %>% mutate(model = "Logistic Regression"),
  rf_auc %>% mutate(model = "Random Forest")
) %>% select(model, .metric, .estimate)


### 💻 Worked Example: Compare Confusion Matrices

For **both** models, we compute a confusion matrix on the test set using `yardstick::conf_mat()`,
and report accuracy, precision, and recall for each. Discuss with the class: which model would
you recommend to the business, and why?


In [ ]:
# ---- Worked Example: Compare Confusion Matrices ----

# Confusion matrix for logistic regression
log_reg_cm <- conf_mat(log_reg_probs, truth = Churn, estimate = .pred_class)
log_reg_cm

# Confusion matrix for the random forest
rf_cm <- conf_mat(rf_probs, truth = Churn, estimate = .pred_class)
rf_cm

# Accuracy, precision, and recall for each model
metric_set_fn <- metric_set(accuracy, precision, recall)

metric_set_fn(log_reg_probs, truth = Churn, estimate = .pred_class, event_level = "second")
metric_set_fn(rf_probs, truth = Churn, estimate = .pred_class, event_level = "second")

---
## Phase 4: Communicating Results & Findings (20 min)

### 📘 Concept

A model is only valuable if its insights reach the people who make decisions. This phase is
about **translation**: turning statistical output (feature importances, coefficients, AUC
scores) into a narrative that a non-technical stakeholder — e.g., a VP of Customer Success —
can act on.

- **Business relevance:** "Our model has an AUC of 0.84" means nothing to a business leader.
  "Customers on month-to-month contracts without tech support are 3x more likely to churn, and
  we can now flag them 60 days before they typically leave" drives action.
- **Explainability tools** like `vip` (Variable Importance Plots) bridge this gap by showing,
  visually, which features the model actually relies on.

**Learning goals:**
1. Extract and visualize variable importance from the Random Forest model.
2. Identify the top business drivers of churn.
3. Practice writing a plain-language executive summary.


In [ ]:
# ---- Variable Importance Plot (Random Forest) ----
rf_fit %>%
  extract_fit_parsnip() %>%
  vip(num_features = 10, geom = "col") +
  labs(title = "Top 10 Drivers of Customer Churn (Random Forest)",
       subtitle = "Ranked by impurity-based importance") +
  theme_minimal()


In [ ]:
# ---- Top 5 drivers, cleanly formatted for a business audience ----
importance_df <- rf_fit %>%
  extract_fit_parsnip() %>%
  vip::vi() %>%
  arrange(desc(Importance)) %>%
  slice_head(n = 5)

importance_df %>%
  ggplot(aes(x = reorder(Variable, Importance), y = Importance)) +
  geom_col(fill = "#2C3E50") +
  coord_flip() +
  labs(
    title = "Top 5 Churn Drivers",
    x = NULL, y = "Relative Importance"
  ) +
  theme_minimal(base_size = 13)


### 💻 Worked Example: Executive Summary

Using the Top-5 driver chart above, here is a **3–5 sentence executive summary** aimed at a VP
of Customer Success. It:
1. States the model's headline finding in plain language (no jargon like "AUC" or "impurity").
2. Recommends one concrete retention action tied to a specific driver.
3. Notes one caveat or limitation the business should be aware of.

**Summary:**

> Our analysis shows that customers on month-to-month contracts, especially those without
> tech support, are far more likely to cancel service than customers on longer-term plans. We
> recommend proactively offering a tech-support add-on or a modest discount to convert
> month-to-month customers onto one-year contracts within their first 90 days, since early
> tenure is when churn risk peaks. One caveat: this model is trained on historical patterns
> and should be re-validated quarterly, since customer behavior and competitor offers change
> over time.

---
## Phase 5: Operationalizing Analytics Models (15 min)

### 📘 Concept

The final phase of the lifecycle is **operationalization**: making the model available for
real-world, repeated use — not just a one-off notebook analysis. This typically involves:

1. **Serialization** — saving the trained pipeline to disk (`.rds`) so it can be reloaded
   without retraining.
2. **Packaging as a service** — exposing the model as an API (in R, via the `plumber` package)
   so other systems (e.g., a CRM) can request real-time predictions.
3. **Monitoring** — tracking model performance over time to catch **concept drift**, i.e., when
   the relationship between features and the outcome changes (e.g., a new competitor changes
   why customers leave), which silently degrades model accuracy.

**Business relevance:** A churn model sitting in a notebook provides zero value. A churn model
wired into the CRM that scores every customer nightly and flags top-risk accounts for the
retention team is what actually reduces churn.

**Learning goals:**
1. Save and reload a trained `tidymodels` workflow.
2. Score a new, unseen record ("production payload").
3. Understand, at a high level, how to expose a model via a `plumber` API.


In [ ]:
# ---- Serialize the trained pipeline ----
saveRDS(rf_fit, "churn_rf_model.rds")

# ---- Reload it (simulating a fresh R session in production) ----
production_model <- readRDS("churn_rf_model.rds")

cat("Model reloaded successfully. Class:", class(production_model)[1], "\n")


In [ ]:
# ---- Simulate a new customer record ("production payload") ----
new_customer <- tibble(
  gender = "Female",
  SeniorCitizen = 0,
  Partner = "No",
  Dependents = "No",
  tenure = 2,
  PhoneService = "Yes",
  MultipleLines = "No",
  InternetService = "Fiber optic",
  OnlineSecurity = "No",
  OnlineBackup = "No",
  DeviceProtection = "No",
  TechSupport = "No",
  StreamingTV = "No",
  StreamingMovies = "No",
  Contract = "Month-to-month",
  PaperlessBilling = "Yes",
  PaymentMethod = "Electronic check",
  MonthlyCharges = 85.4,
  TotalCharges = 170.8
)

# ---- Generate a real-time churn risk score ----
risk_score <- predict(production_model, new_customer, type = "prob")

cat("Predicted probability of churn for this customer:",
    round(risk_score$.pred_Yes, 3), "\n")


### 💻 Packaging as an API with `plumber`

Below is a **template** for a standalone `plumber.R` script. This is not meant to be run
inside the notebook — it's what you would save as a separate file and deploy so that any
system (e.g., a CRM) can request a churn score over HTTP.


In [ ]:
# ==========================================================
# plumber.R  -- save this as its own file and run with:
#   library(plumber)
#   pr("plumber.R") %>% pr_run(port = 8000)
# ==========================================================
#
# library(plumber)
#
# # Load the serialized model once, at API startup
# model <- readRDS("churn_rf_model.rds")
#
# #* Predict churn risk for a single customer
# #* @param gender:character
# #* @param tenure:numeric
# #* @param Contract:character
# #* @param MonthlyCharges:numeric
# #* @param TotalCharges:numeric
# #* ... (additional parameters mirror the model's feature columns)
# #* @post /predict_churn
# function(gender, tenure, Contract, MonthlyCharges, TotalCharges, ...) {
#   new_data <- tibble::tibble(
#     gender = gender,
#     tenure = as.numeric(tenure),
#     Contract = Contract,
#     MonthlyCharges = as.numeric(MonthlyCharges),
#     TotalCharges = as.numeric(TotalCharges)
#     # ... remaining fields populated from request body / defaults
#   )
#   prob <- predict(model, new_data, type = "prob")
#   list(churn_probability = prob$.pred_Yes)
# }
#
# ==========================================================
# In production, this endpoint would be called nightly (or in
# real time) by the CRM to score every active customer, and the
# top-N highest-risk accounts would be routed to the retention team.
# ==========================================================

cat("Plumber API template printed above — save separately as plumber.R to deploy.\n")


### 🩺 A Note on Monitoring & Concept Drift

Once deployed, the model's job isn't done. Best practice is to:
- Log every prediction alongside the eventual real-world outcome (did the customer actually churn?).
- Periodically recompute AUC / recall on fresh data and compare to the original test-set metrics.
- Set an alert threshold (e.g., "if AUC drops more than 0.05 from baseline, trigger retraining").

This closes the loop of the Big Data Analytics Lifecycle — insights lead to action, and action
generates new data that feeds the next iteration of the cycle.

---

## 🎉 Workshop Wrap-Up

You've now walked through all five phases of the Big Data Analytics Lifecycle:

1. ✅ **Data Preparation** — cleaned and encoded a messy real-world dataset
2. ✅ **Model Planning & EDA** — formed hypotheses backed by correlation and visualization
3. ✅ **Model Building & Validation** — trained and compared two classification models
4. ✅ **Communicating Results** — translated feature importance into business language
5. ✅ **Operationalizing** — serialized a model and sketched an API for production use

**Next steps for further practice:**
- Try tuning the random forest's hyperparameters with `tune::tune_grid()`.
- Explore `step_upsample()` / `step_downsample()` (via the `themis` package) to address class imbalance.
- Extend the `plumber` template into a fully working local API.

Thanks for participating! 🎓
